In [ ]:
import dpsimpy  
import numpy 

In [ ]:
print(dir(dpsimpy.emt.ph1))
help(dpsimpy.emt.ph1)

In [ ]:
# Nodes
gnd = dpsimpy.emt.SimNode.gnd
n1  = dpsimpy.emt.SimNode("n1")
n2  = dpsimpy.emt.SimNode("n2")
n3  = dpsimpy.emt.SimNode("n3")
n4  = dpsimpy.emt.SimNode("n4")
n5  = dpsimpy.emt.SimNode("n5")
n6  = dpsimpy.emt.SimNode("n6")
n7  = dpsimpy.emt.SimNode("n7")
n8  = dpsimpy.emt.SimNode("n8")
n9  = dpsimpy.emt.SimNode("n9")
n10 = dpsimpy.emt.SimNode("n10")
n11 = dpsimpy.emt.SimNode("n11")
n12 = dpsimpy.emt.SimNode("n12")
n13 = dpsimpy.emt.SimNode("n13")
n14 = dpsimpy.emt.SimNode("n14")

# Components

v_a = dpsimpy.emt.ph1.VoltageSource("v_a")
v_a.set_parameters(complex(10, 0), 50)
v_b = dpsimpy.emt.ph1.VoltageSource("v_b")
v_b.set_parameters(complex(-5, -5*numpy.sqrt(3.0)), 50)
v_c = dpsimpy.emt.ph1.VoltageSource("v_c")
v_c.set_parameters(complex(-5, 5*numpy.sqrt(3.0)), 50)

r1 = dpsimpy.emt.ph1.Resistor("r1")
r1.set_parameters(1)
r2 = dpsimpy.emt.ph1.Resistor("r2")
r2.set_parameters(1)
r3 = dpsimpy.emt.ph1.Resistor("r3")
r3.set_parameters(1)

clarke_transformer = dpsimpy.emt.ph1.ClarkeTransformer("clarke_transformer")

r4 = dpsimpy.emt.ph1.Resistor("r4")
r4.set_parameters(50)
r5 = dpsimpy.emt.ph1.Resistor("r5")
r5.set_parameters(50)
r6 = dpsimpy.emt.ph1.Resistor("r6")
r6.set_parameters(50)

r_neutral = dpsimpy.emt.ph1.Resistor("r_neutral")
r_neutral.set_parameters(3*(r1.R+r4.R)) # Neutral resistance is 3 times the phase resistance

l1 = dpsimpy.emt.ph1.Inductor("l1")
l1.set_parameters(0.1)
l2 = dpsimpy.emt.ph1.Inductor("l2")
l2.set_parameters(0.1)
l3 = dpsimpy.emt.ph1.Inductor("l3")
l3.set_parameters(0.1)

# Connections

v_a.connect([gnd, n1])
v_b.connect([gnd, n2])
v_c.connect([gnd, n3])
r1.connect([n1, n4])
r2.connect([n2, n5])
r3.connect([n3, n6])
clarke_transformer.connect([n4, n5, n6, n7, n8, n9])
r4.connect([n7, n10])
r5.connect([n8, n11])
r6.connect([n9, n12])
l1.connect([n10, gnd])
l2.connect([n11, gnd])
l3.connect([n12, n13])
r_neutral.connect([n13, gnd])



In [ ]:
sys = dpsimpy.SystemTopology(50, [gnd, n1, n2, n3, n4, n5, n6, n7, n8, n9, n10, n11, n12, n13], [v_a, v_b, v_c, r1, r2, r3, clarke_transformer, r4, r5, r6, l1, l2, l3, r_neutral])
sys

In [ ]:
sim = dpsimpy.Simulation("Clarke Transformer", loglevel=dpsimpy.LogLevel.debug)
sim.set_system(sys)
sim.set_domain(dpsimpy.Domain.EMT)
sim.set_time_step(0.00005)
sim.set_final_time(0.2)
sim.do_eigenvalue_extraction(True)

abc_nodes = [n4, n5, n6]
clarke_nodes = [n7, n8, n9]

log = dpsimpy.Logger("Clarke Transformer")
#for i in range(0, len(sys.nodes)):
#    log.log_attribute("v" + str(i), "v", sys.nodes[i])

# Log voltages for ABC side (naming as v1, v2, v3)
for i, node in enumerate(abc_nodes):
    log.log_attribute(f"v{i+1}", "v", node)

# Log voltages for alpha-beta-zero domain using fixed names
log.log_attribute("v_alpha", "v", clarke_nodes[0])
log.log_attribute("v_beta", "v", clarke_nodes[1])
log.log_attribute("v_0", "v", clarke_nodes[2])

sim.add_logger(log)
    
sim.run()

In [ ]:
%matplotlib inline
%config InlineBackend.figure_format = 'svg'
%config InlineBackend.rc = {'font.size': 10, 'figure.figsize': (6.0, 4.0), 'figure.facecolor': 'white', 'savefig.dpi': 72, 'figure.subplot.bottom': 0.125, 'figure.edgecolor': 'white'}

import matplotlib.pyplot as plt
import villas.dataprocessing.plottools as pt
import villas.dataprocessing.readtools as rt
import villas.dataprocessing.timeseries as ts

results     = rt.read_timeseries_dpsim('logs/Clarke Transformer.csv')
results_emt = [ results[series].frequency_shift(freq=0) for series in results ]

#LOOK AT THE REAL COLUMN NAMES TO PLOT THE RIGHT VOLTAGE

#for series in results_emt:
    #pt.plot_timeseries('Results EMT', series)
pt.plot_timeseries('Results EMT', results_emt[3])
plt.show()